In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("DeltaTables_Incremental_loading").getOrCreate()


In [0]:
%sql
select * from unity_cata_workspace.source.products

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

df_src = spark.sql("select * from unity_cata_workspace.source.products")

# dedeuplicate dataframe

frame = Window.partitionBy("p_id").orderBy(desc("update_time"))
df_src = df_src.withColumn('rn', row_number().over(frame)).filter(col('rn')==1).drop('rn')


df_src.display()

### UPSERT

In [0]:
from delta.tables import DeltaTable



if len(dbutils.fs.ls("/Volumes/unity_cata_workspace/source/db_volumne/product_sink/"))>0:
# try:
    dlt_obj = DeltaTable.forPath(spark,"/Volumes/unity_cata_workspace/source/db_volumne/product_sink/")

    dlt_obj.alias('trg').merge(
        df_src.alias('src'),
        "trg.p_id=src.p_id"
    ).whenMatchedUpdateAll(condition="src.update_time > trg.update_time")\
    .whenNotMatchedInsertAll()\
    .execute()
# except:
else:
    df_src.write.format("delta").mode("overwrite")\
        .save("/Volumes/unity_cata_workspace/source/db_volumne/product_sink/")


In [0]:
%sql
select * from delta.`/Volumes/unity_cata_workspace/source/db_volumne/product_sink/`  